In [ ]:
# Import packages 
import os
import sys
workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature




from myPythonTools.TopoAna import Plots as Po
from myPythonTools.TopoAna import rdtilemap as Ti
from myPythonTools.TopoAna import rdremap as RM
from myPythonTools.TopoAna import rdridgelist as RL


import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.tri as tri
import matplotlib.colors as colors

from scipy.io import FortranFile

import importlib
import glob
import copy

importlib.reload(Po)
importlib.reload(Ti)
importlib.reload(RM)
importlib.reload(RL)

#sys.path.append('../Plotting/')
#   Now you can imprt modules in ../Plotting
#import xyp_plot as xyp
#import ana as a


In [ ]:
caseN = 'ERA5-era5-2016-11-Rw050-x02'

fileN = f'/glade/derecho/scratch/juliob/archive/GW_UnitTest/{caseN}/{caseN}.h.2016-11-29-00000.nc'
#fileN = f'/glade/derecho/scratch/juliob/archive/GW_UnitTest/{caseN}/{caseN}.h.2016-11-01-00000.nc'


X=xr.open_dataset( fileN )

X.bnd_topo


#----------------------------------------
# get remappers and WRF CONUS404 data
from cs_wrf_mapper import CS_WRF_Mapper
from se_wrf_mapper import SE_WRF_Mapper


f1='/glade/campaign/collections/rda/data/d559000/wy2017/201611/wrf2d_d01_2016-11-29_00:00:00.nc'
#f1='/glade/campaign/collections/rda/data/d559000/wy2017/201611/wrf2d_d01_2016-11-01_00:00:00.nc'

W=xr.open_dataset( f1 )
xlong=W.XLONG.values
xlat=W.XLAT.values
xlong_u=W.XLONG_U.values
xlat_u=W.XLAT_U.values
xlong_v=W.XLONG_V.values
xlat_v=W.XLAT_V.values


In [ ]:
tau_a = X.TAU_A_RDG.values
tau_b = X.TAU_B_RDG.values
taudsw = X.TAUDSW.values
tauoro = X.TAUORO.values

sgh=X.SGH.values
lonc=X.lon.values
latc=X.lat.values

usrc = X.USRC_RDG.values
vsrc = X.VSRC_RDG.values


# ---------------------------------
# Basic wave-related quantities
# consistent with rdg GW scheme.
#----------------------------------
nsrc=X.NSRC_RDG.values
ubsrc=X.UBMSRC_RDG.values
hwdth=1000.*X.HWDTH_RDG.values 
mxd=X.MXDIS_RDG

mxd.shape
ktild=1./(hwdth+1.)
rho=1.0
delprm2 = hwdth*taudsw /(rho*nsrc*ubsrc)
delprm = np.sqrt( delprm2 )
uprm = nsrc*delprm
utot= ubsrc+uprm
#warmy=delprm*9.8/1000.
Tsrc=280.
warmy = delprm * nsrc**2 * Tsrc / 9.8

delprm2_oro = hwdth*tauoro /(rho*nsrc*ubsrc)
delprm_oro = np.sqrt( delprm2_oro )

uprm_dsw = nsrc*(delprm - delprm_oro )

# Sanity check

fig,axs=plt.subplots( 1,5, figsize=( 5*8, 7 ) )

p=0
ax=axs[p]
ax.plot( delprm[0,0,:] )
ax.set_title( 'displacment (m)' )

p=p+1
ax=axs[p]
ax.plot( uprm[0,0,:] )
ax.set_title( ' up (m/s)' )

p=p+1
ax=axs[p]
ax.plot( warmy[0,0,:] )
ax.set_title( ' Tp (K)' )

p=p+1
ax=axs[p]
ax.plot( utot[0,0,:] )
ax.set_title( ' U_tot (m/s)' )

p=p+1
ax=axs[p]
ax.plot( uprm_dsw[0,0,:] )
ax.set_title( ' Up_DSW (m/s)' )



In [ ]:
mxd.shape

In [ ]:
import paint_tiles_nc as Pti
importlib.reload( Pti )


from paint_tiles_nc import load_tiles_nc, paint_tiles_nc

filn = X.bnd_topo  #'/glade/work/juliob/Topo/NCARTopoJTB-fire//cases/ne30pg3_Sco100_v202_rch1p0_e01/output/ne30pg3_gmted2010_modis_bedmachine_nc3000_Laplace0100_noleak_greenlndantarcsgh30fac2.50_20260907_tiles.nc'

tg = Pti.load_tiles_nc( filn, ncube=3000, grid_length_scale=3.3)
A  = (taudsw - tauoro)[0,:,:] # (ncol, nrdg)
A  = warmy[0,:,:] # (ncol, nrdg)

field_warm = Pti.paint_tiles_nc( warmy[0,:,:]   , tg, shape='rectangle',  width_scale=1.0, verbose=True, ir_range=(0,43) ,  side='downwind', usrc=usrc[0,:,:], vsrc=vsrc[0,:,:]   , modulation='ramp_downwind')
field_uprm = Pti.paint_tiles_nc( uprm[0,:,:]   , tg, shape='rectangle',  width_scale=1.0, verbose=True, ir_range=(0,43) ,  side='downwind', usrc=usrc[0,:,:], vsrc=vsrc[0,:,:]   , modulation='cosine')
field_uprm_dsw = Pti.paint_tiles_nc( uprm_dsw[0,:,:]   , tg, shape='rectangle',  width_scale=1.0, verbose=True, ir_range=(0,43) ,  side='downwind', usrc=usrc[0,:,:], vsrc=vsrc[0,:,:]   , modulation='ramp_downwind')
field_mxd = Pti.paint_tiles_nc(mxd[0,:,:], tg, shape='rectangle',  width_scale=0.1, verbose=True, ir_range=(0,43) )


In [ ]:
#------------------------------------
# Quickie plot

fig,ax=plt.subplots(figsize=(7,5) )

#plt.contourf( field[3,:,:] , cmap='plasma', levels= np.linspace(0.1,1.1,11) ,alpha=0.8)
#ax.contourf( field2[3,:,:] , colors='black', levels= 51 )
c1 = ax.contourf( field_uprm_dsw[3,:,:] , cmap='OrRd', levels=   np.linspace(0.1,30.1,21) ) #  np.linspace(0.1,1.1,11) ) #   np.linspace(1,31,41) ,alpha=1)
ax.set_xlim( 1500,2000 )
ax.set_ylim( 2500,3000 )
plt.colorbar( c1 )

In [ ]:
#--------------------------------------
#  Get 3000x3000x6 topo stuff

from pathlib import Path

topo_dir = str(Path(X.bnd_topo).parent)
print(topo_dir)

#print( filn )

foo=glob.glob( f"{topo_dir}/topo*.nc")
FileN = foo[-1]
print(FileN)
topo=xr.open_dataset( FileN )
print(list(topo.variables))
print(topo.attrs)
raw=topo['terr_dev']+topo['terr_sm']
Loo = np.size(raw)
nc=int(np.sqrt( Loo/6 ))
print(int(nc))
raw=np.reshape( raw.values , (6,nc,nc) )
dev=np.reshape( topo['terr_dev'].values , (6,nc,nc) )
smoo=np.reshape( topo['terr_sm'].values , (6,nc,nc) )
tlats=np.reshape( topo['lat'].values , (6,nc,nc) )
tlons=np.reshape( topo['lon'].values , (6,nc,nc) )



In [ ]:
%%time 
#------------------------------------
# MAke mappers to WRF CONUS404 grid

mapper_cs = CS_WRF_Mapper(
    tlons,
    tlats,
    xlong,
    xlat,
)


mapper_se = SE_WRF_Mapper(
    lonc,
    latc,
    xlong,
    xlat,
)

In [ ]:
import WRF_utils as Wu

us = W.U.values
vs = W.V.values
uwrf,vwrf=Wu.uv_to_mass(us, vs)
us.shape


In [ ]:
field_warm=np.nan_to_num(field_warm)
field_uprm=np.nan_to_num(field_uprm)

te=X.T.values
cwsp = np.sqrt( X.U.values**2 + X.V.values**2 )
wspwrf = np.sqrt( uwrf**2 + vwrf**2 )
te_wrf = mapper_se.se_to_wrf( te[0,92,:] )
cwsp_wrf = mapper_se.se_to_wrf( cwsp[0,92,:] )

topo_wrf=mapper_cs.cs_to_wrf( raw )
devtopo_wrf=mapper_cs.cs_to_wrf( dev )
warm_wrf=mapper_cs.cs_to_wrf( field_warm )
uprm_wrf=mapper_cs.cs_to_wrf( field_uprm )


uprm_wrf = np.nan_to_num(uprm_wrf, nan=0.0)
warm_wrf = np.nan_to_num(warm_wrf, nan=0.0)
topo_wrf = np.nan_to_num(topo_wrf, nan=0.0)
devtopo_wrf = np.nan_to_num(devtopo_wrf, nan=0.0)


In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

data_crs = ccrs.PlateCarree()

proj=ccrs.PlateCarree()
"""
proj = ccrs.LambertConformal(
    central_longitude=-97.9,
    central_latitude=39.1,
    standard_parallels=(30., 50.)
)
"""
wsplvs = np.linspace( 0,30,num=41 )

cmapN='OrRd'

xy = proj.transform_points(data_crs, xlong, xlat)
xx = xy[..., 0]
yy = xy[..., 1]

xmin, xmax = np.nanmin(xx), np.nanmax(xx)
ymin, ymax = np.nanmin(yy), np.nanmax(yy)


fields = [
    cwsp_wrf,
    cwsp_wrf + 1.5*uprm_wrf,
    wspwrf[0, :, :],
]

titles = [
    'Mapped CAM-SE wind speed',
    'Mapped CAM-SE wind speed + mountain wave enhancement',
    'WRF CONUS wind speed',
]

fig, axs = plt.subplots(
    1, 3,
    figsize=(20, 7),
    subplot_kw={'projection': proj},
    constrained_layout=True
)

for ax, fld, title in zip(axs, fields, titles):

    cf = ax.contourf(
        xlong, xlat,
        np.ma.masked_invalid(fld),
        levels=wsplvs,
        cmap=cmapN,
        extend='max',
        transform=data_crs
    )

    ax.coastlines(resolution='50m', linewidth=0.7)

    ax.add_feature(
        cfeature.STATES,
        linewidth=0.35,
        edgecolor='0.5'
    )

    ax.add_feature(
        cfeature.BORDERS,
        linewidth=0.5
    )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    ax.set_title(title, fontsize=13)

cbar = fig.colorbar(
    cf,
    ax=axs,
    orientation='horizontal',
    shrink=0.40,
    pad=0.04
)

cbar.set_label(r'Wind speed (m s$^{-1}$)')

fig.suptitle(
    f'Comparison of WRF and remapped CAM-SE winds {caseN}',
    fontsize=15
)

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

data_crs = ccrs.PlateCarree()

proj=ccrs.PlateCarree()
"""
proj = ccrs.LambertConformal(
    central_longitude=-97.9,
    central_latitude=39.1,
    standard_parallels=(30., 50.)
)
"""
wsplvs = np.linspace( 0,30,num=41 )

cmapN='OrRd'

xy = proj.transform_points(data_crs, xlong, xlat)
xx = xy[..., 0]
yy = xy[..., 1]

xmin, xmax = -90., -80. #np.nanmin(xx), np.nanmax(xx)
ymin, ymax = 34.,38. #np.nanmin(yy), np.nanmax(yy)


fields = [
    cwsp_wrf,
    cwsp_wrf + 1.5*uprm_wrf,
    wspwrf[0, :, :],
]

titles = [
    'Mapped CAM-SE wind speed',
    'Mapped CAM-SE wind speed + mountain wave enhancement',
    'WRF CONUS wind speed',
]

fig, axs = plt.subplots(
    1, 3,
    figsize=(20, 7),
    subplot_kw={'projection': proj},
    constrained_layout=True
)

for ax, fld, title in zip(axs, fields, titles):

    cf = ax.contourf(
        xlong, xlat,
        np.ma.masked_invalid(fld),
        levels=wsplvs,
        cmap=cmapN,
        extend='max',
        transform=data_crs
    )

    ax.coastlines(resolution='50m', linewidth=0.7)

    ax.add_feature(
        cfeature.STATES,
        linewidth=0.35,
        edgecolor='0.5'
    )

    ax.add_feature(
        cfeature.BORDERS,
        linewidth=0.5
    )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    ax.set_title(title, fontsize=13)

cbar = fig.colorbar(
    cf,
    ax=axs,
    orientation='horizontal',
    shrink=0.40,
    pad=0.04
)

cbar.set_label(r'Wind speed (m s$^{-1}$)')

fig.suptitle(
    'Comparison of WRF and remapped CAM-SE winds',
    fontsize=15
)

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

data_crs = ccrs.PlateCarree()

proj=ccrs.PlateCarree()
"""
proj = ccrs.LambertConformal(
    central_longitude=-97.9,
    central_latitude=39.1,
    standard_parallels=(30., 50.)
)
"""
wsplvs = np.linspace( 0,30,num=41 )

cmapN='OrRd'

xy = proj.transform_points(data_crs, xlong, xlat)
xx = xy[..., 0]
yy = xy[..., 1]

xmin, xmax = np.nanmin(xx), np.nanmax(xx)
ymin, ymax = np.nanmin(yy), np.nanmax(yy)


fields = [
    devtopo_wrf,
    cwsp_wrf + 1.5*uprm_wrf,
    wspwrf[0, :, :],
]

titles = [
    'Mapped CAM-SE wind speed',
    'Mapped CAM-SE wind speed + mountain wave enhancement',
    'WRF CONUS wind speed',
]

clevls = [ np.linspace(-1000,1000, num=41) , wsplvs, wsplvs ]

fig, axs = plt.subplots(
    1, 3,
    figsize=(20, 7),
    subplot_kw={'projection': proj},
    constrained_layout=True
)

for ax, fld, title, clev in zip(axs, fields, titles, clevls ):

    mask = np.abs(devtopo_wrf) > 300.

    fld = np.ma.masked_where(~mask, fld)

 
    cf = ax.contourf(
        xlong, xlat,
        np.ma.masked_invalid(fld),
        levels=clev,
        cmap=cmapN,
        extend='max',
        transform=data_crs
    )

    ax.coastlines(resolution='50m', linewidth=0.7)

    ax.add_feature(
        cfeature.STATES,
        linewidth=0.35,
        edgecolor='0.5'
    )

    ax.add_feature(
        cfeature.BORDERS,
        linewidth=0.5
    )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    ax.set_title(title, fontsize=13)

cbar = fig.colorbar(
    cf,
    ax=axs,
    orientation='horizontal',
    shrink=0.40,
    pad=0.04
)

cbar.set_label(r'Wind speed (m s$^{-1}$)')

fig.suptitle(
    'Comparison of WRF and remapped CAM-SE winds',
    fontsize=15
)

In [ ]:
print(ymin,ymax)

In [ ]:
from scipy.ndimage import gaussian_filter

#smoothed = gaussian_filter(field, sigma=2)  # sigma in grid points
sigma=3
sigma_ls=30

"""
u0 = gaussian_filter(
    cwsp_wrf - gaussian_filter(cwsp_wrf, sigma=sigma_ls),
    sigma=sigma
).flatten()

u1 = gaussian_filter(
    cwsp_wrf + 1.5*uprm_wrf
    - gaussian_filter(cwsp_wrf + 1.5*uprm_wrf, sigma=sigma_ls),
    sigma=sigma
).flatten()

u2 = gaussian_filter(
    wspwrf[0, :, :]
    - gaussian_filter(wspwrf[0, :, :], sigma=sigma_ls),
    sigma=sigma
).flatten()

up1 = gaussian_filter(
    uprm_wrf,
    sigma=sigma
).flatten()
"""
print( devtopo_wrf.shape )
print( cwsp_wrf.shape )

wspwrf0 = wspwrf[0, :, :]
print( wspwrf[0,:,:].shape )


sigma=5
cwsp_wrf_s = gaussian_filter( cwsp_wrf, sigma=sigma )
uprm_wrf_s = gaussian_filter( uprm_wrf, sigma=sigma )
wspwrf0_s = gaussian_filter( wspwrf0, sigma=sigma )
devtopo_s = gaussian_filter( devtopo_wrf, sigma=sigma )

maskT = np.abs(devtopo_s) > 300.
cwsp_wrf_m= np.ma.masked_where(~maskT, cwsp_wrf_s)
uprm_wrf_m= np.ma.masked_where(~maskT, uprm_wrf_s)
wspwrf0_m= np.ma.masked_where(~maskT, wspwrf0_s)

In [ ]:
plt.contour( devtopo_s )
#plt.plot( uprm_wrf_s[600:700,400] )


In [ ]:
x0 = np.ma.filled(cwsp_wrf_m, np.nan).flatten()
x1 = np.ma.filled(cwsp_wrf_m + 1.5*uprm_wrf_m, np.nan).flatten()
y  = np.ma.filled(wspwrf0_m, np.nan).flatten()

good = np.isfinite(x0) & np.isfinite(x1) & np.isfinite(y)

x0 = x0[good]
x1 = x1[good]
y  = y[good]

def stats(x, y):
    r    = np.corrcoef(x, y)[0,1]
    bias = np.mean(x - y)
    rmse = np.sqrt(np.mean((x - y)**2))
    slope, intercept = np.polyfit(x, y, 1)

    return r, bias, rmse, slope, intercept

print("No uprm :", stats(x0, y))
print("With uprm:", stats(x1, y))

In [ ]:
plt.figure(figsize=(7,7))

plt.scatter(x1, y, s=.4, alpha=.2, label='CAM-SE + 1.5 uprm')
plt.scatter(x0, y, s=.4, alpha=.2, label='CAM-SE')

vmax = max(x0.max(), x1.max(), y.max())

plt.plot([0,vmax], [0,vmax], 'k--', lw=1.5)

plt.xlim(0, vmax)
plt.ylim(0, vmax)
plt.gca().set_aspect('equal')

plt.xlabel('CAM-SE-derived wind speed (m/s)')
plt.ylabel('WRF wind speed (m/s)')
plt.legend()

In [ ]:
plt.scatter( x0, y , s=.1 )
plt.scatter( x1, y , s=.1 )



In [ ]:
print( stop_here_now )

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

data_crs = ccrs.PlateCarree()

proj = ccrs.LambertConformal(
    central_longitude=-97.9,
    central_latitude=39.1,
    standard_parallels=(30., 50.)
)

xy = proj.transform_points(data_crs, xlong, xlat)
xx = xy[..., 0]
yy = xy[..., 1]

xmin, xmax = np.nanmin(xx), np.nanmax(xx)
ymin, ymax = np.nanmin(yy), np.nanmax(yy)

fields = [
    wspwrf[0, :, :],
    cwsp_wrf + uprm_wrf,
    cwsp_wrf
]

titles = [
    'WRF wind speed',
    'Mapped CAM-SE wind + uprm',
    'Mapped CAM-SE wind'
]

fig, axs = plt.subplots(
    1, 3,
    figsize=(20, 7),
    subplot_kw={'projection': proj},
    constrained_layout=True
)

for ax, fld, title in zip(axs, fields, titles):

    cf = ax.contourf(
        xlong, xlat,
        np.ma.masked_invalid(fld),
        levels=wsplvs,
        cmap='viridis',
        extend='max',
        transform=data_crs
    )

    ax.coastlines(resolution='50m', linewidth=0.7)

    ax.add_feature(
        cfeature.STATES,
        linewidth=0.35,
        edgecolor='0.5'
    )

    ax.add_feature(
        cfeature.BORDERS,
        linewidth=0.5
    )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    ax.set_title(title, fontsize=13)

cbar = fig.colorbar(
    cf,
    ax=axs,
    orientation='horizontal',
    shrink=0.80,
    pad=0.04
)

cbar.set_label(r'Wind speed (m s$^{-1}$)')

fig.suptitle(
    'Comparison of WRF and remapped CAM-SE winds',
    fontsize=15
)

In [ ]:
field_warm.shape
plt.plot( field_warm[3,2500,:] )

In [ ]:
print(warm_wrf.shape)
print(te_xrf.shape)

plt.plot( topo_wrf[500,:] )


In [ ]:
import glob
import os
import xarray as xr
import numpy as np

outdir = TopoRoot + 'cases/' + Case + '/output'

# Find ordinary files, explicitly excluding *_tiles.nc
foo = [
    f for f in glob.glob(outdir + '/'+ ogrid + '*.nc')
    if not f.endswith('_tiles.nc')
]

# Take newest file
FileN = max(foo, key=os.path.getmtime)

# Corresponding tiles file
FileTiles = FileN.replace('.nc', '_tiles.nc')

if not os.path.exists(FileTiles):
    raise FileNotFoundError(f"No corresponding tiles file:\n{FileTiles}")

print("Main file: ", FileN)
print("Tiles file:", FileTiles)
print("Topo file: ", X.bnd_topo )

# Main grid file
dCAM = xr.open_dataset(FileN)

print(list(dCAM.variables))
print(np.shape(dCAM['lat']))

target_lon  = dCAM['lon'].values
target_lat  = dCAM['lat'].values
target_topo = dCAM['PHIS'].values / 9.8

# Tile file
dTiles = xr.open_dataset(FileTiles)

print(list(dTiles.variables))

In [ ]:
plt.plot( field[0,:,:].flatten() )

In [ ]:
field.shape

In [ ]:
taudsw.shape

In [ ]:
#co = plt.tricontourf( lonc,latc, np.sum( tau_b-0*tau_a, axis=(0,1,2) )[:] )
co = plt.tricontourf( lonc,latc, np.sum( taudsw - tauoro, axis=(0,1) )[:], cmap='plasma', levels=np.linspace(0,10,11) )
li=plt.tricontour( lonc,latc, sgh, levels=[-1,0,1,10000] ,colors='w' )
#plt.xlim(240,250)
#plt.ylim(30,36)
plt.xlim(270,300)
plt.ylim(30,38)
plt.xlabel('Lon')
plt.ylabel('Lat')
plt.title( 'Nov 28 2016 ' )
plt.colorbar(co)

In [ ]:
plt.plot( taudsw[0,0,:] )
plt.plot( tauoro[0,0,:] )


In [ ]:
nsrc=X.NSRC_RDG.values
ubsrc=X.UBMSRC_RDG.values

hwdth=1000.*X.HWDTH_RDG.values 

In [ ]:
plt.plot( hwdth[0,0,:] /1000. )

In [ ]:
plt.plot( ubsrc[0,0,:] ,'.')



uprm=hwdth * np.sqrt( taudsw ) / ubsrc 
delprm=hwdth * np.sqrt( taudsw ) / (ubsrc *nsrc)

In [ ]:
plt.plot((ubsrc)[0,0,:] )
#plt.plot((uprm)[0,0,:] )
utot=uprm+ubsrc

In [ ]:
#plt.plot((9.8/1000.)*(delprm)[0,0,:] ) 
warmy = (9.8/1000.)*delprm
plt.plot(warmy[0,0,:] ) 

In [ ]:
X

In [ ]:
from regrid_to_cube import read_cube_target_map, regrid_to_cube

itrgt = read_cube_target_map('/glade/work/juliob/Topo/NCARTopoJTB-fire/cases/ne30pg3_Sco100_v202_rch1p0_e01/output/grid_remap_nc3000_ne30pg3.dat', verbose=True)
#cube  = regrid_to_cube(taudsw_column_sum, itrgt)   # -> (6, 3000, 3000)

In [ ]:
te=X.T.values
te_cube  = regrid_to_cube(te[0,92,:], itrgt)  



In [ ]:
plt.contourf( te_cube[3,:,:] )

In [ ]:


import numpy as np
from paint_tiles_nc import load_tiles_nc, paint_tiles_nc

tg = load_tiles_nc('..._tiles.nc', ncube=3000, grid_length_scale=3.336)

def km_decay(s, t, L, W):
    """Exponential decay with a 25 km e-folding scale."""
    return np.exp(-np.abs(t) * 3.336 / 25.0)

field = paint_tiles_nc(A, tg, shape='triangle', side='downwind',
                       usrc=usrc, vsrc=vsrc, modulation=km_decay)